# Colab T4 Benchmark — Tez Aşama 0

Bu notebook bilimsel sonuç üretmez. Amaç, tez deneylerinden önce gerçek NVIDIA T4 ortamını, epoch sürelerini ve GPU bellek kullanımını ölçmektir.

**Önce:** `Runtime > Change runtime type > T4 GPU` seçin.


In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)


## GitHub erişimi

Repo özel olduğu için Colab'ın sol kenarındaki **anahtar (Secrets)** bölümüne yalnızca repo okuma yetkisi olan bir GitHub token ekleyin ve adını `GITHUB_TOKEN` yapın. Token notebook içine yazılmaz ve çıktıya basılmaz.


In [ ]:
from google.colab import userdata
import requests, tarfile, io, shutil, os
from pathlib import Path

OWNER = 'erdemci07'
REPO = 'derin-ogrenme-egitim-dinamikleri'
BRANCH = 'main'
TOKEN = userdata.get('GITHUB_TOKEN')
if not TOKEN:
    raise RuntimeError('Colab Secrets içinde GITHUB_TOKEN bulunamadı.')

headers = {
    'Authorization': f'Bearer {TOKEN}',
    'Accept': 'application/vnd.github+json',
    'X-GitHub-Api-Version': '2022-11-28',
}

commit_resp = requests.get(
    f'https://api.github.com/repos/{OWNER}/{REPO}/commits/{BRANCH}',
    headers=headers,
    timeout=60,
)
commit_resp.raise_for_status()
commit_sha = commit_resp.json()['sha']

archive_resp = requests.get(
    f'https://api.github.com/repos/{OWNER}/{REPO}/tarball/{commit_sha}',
    headers=headers,
    timeout=120,
)
archive_resp.raise_for_status()

extract_root = Path('/content/_tez_repo_extract')
repo_dir = Path('/content/derin-ogrenme-egitim-dinamikleri')
shutil.rmtree(extract_root, ignore_errors=True)
shutil.rmtree(repo_dir, ignore_errors=True)
extract_root.mkdir(parents=True, exist_ok=True)

with tarfile.open(fileobj=io.BytesIO(archive_resp.content), mode='r:gz') as tar:
    tar.extractall(extract_root)

kokler = [p for p in extract_root.iterdir() if p.is_dir()]
if len(kokler) != 1:
    raise RuntimeError('GitHub arşivi beklenen klasör yapısında değil.')
shutil.move(str(kokler[0]), str(repo_dir))

os.environ['TEZ_GIT_COMMIT'] = commit_sha
os.chdir(repo_dir)
print('Repo hazır.')
print('Commit:', commit_sha)


## Sonuçların Google Drive'a kalıcı kaydı


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CIKTI = Path('/content/drive/MyDrive/tez_deney_sonuclari/benchmark_colab_t4')
DRIVE_CIKTI.mkdir(parents=True, exist_ok=True)
print('Çıktı klasörü:', DRIVE_CIKTI)


## Colab bağımlılıkları

Colab'ın kendi CUDA destekli PyTorch kurulumu korunur; bu nedenle `requirements_colab.txt` içinde torch/torchvision yeniden kurulmaz.


In [ ]:
subprocess.run([
    'python', '-m', 'pip', 'install', '-q', '-r', 'requirements_colab.txt'
], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA kullanılabilir:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA aktif değil. Runtime ayarından T4 GPU seçin.')


## Benchmarkı çalıştır

1 warmup + 3 ölçüm epochu CNN-v1 ve CIFAR uyarlanmış ResNet-18 için çalıştırılır.


In [ ]:
cikti = DRIVE_CIKTI / 'benchmark_t4.json'
subprocess.run([
    'python', 'benchmark_colab_t4.py',
    '--warmup-epoch', '1',
    '--olcum-epoch', '3',
    '--batch-size', '128',
    '--num-workers', '2',
    '--seed', '42',
    '--cikti', str(cikti),
], check=True, env=os.environ.copy())


## Özeti göster


In [ ]:
import json
veri = json.loads(cikti.read_text(encoding='utf-8'))
print('GPU:', veri['ortam']['gpu'])
print('Commit:', veri['ortam']['git_commit'])
for sonuc in veri['benchmark']:
    print(
        sonuc['model'],
        'ortalama epoch =', round(sonuc['mean_total_seconds_per_epoch'], 2), 'sn',
        '| tepe bellek =', round(sonuc['peak_gpu_memory_mb'], 0), 'MB'
    )
print('\nKaba kapasite tahminleri:')
for k, v in veri['kaba_kapasite_tahminleri'].items():
    if isinstance(v, (int, float)):
        print('-', k, '=', round(v, 2), 'GPU-saat')
print('\nDosya:', cikti)


## Bu aşamadan sonra

`benchmark_t4.json` dosyasındaki gerçek süreler kullanılarak 150, 300 ve maksimum 1.035 koşuluk planın uygulanabilirliği hesaplanacaktır. Bu notebook çıktıları tez bulgusu değildir.
